# Phase 1: Event-Level Data Merge

Goal: Merge LogFile, UsnJrnl, and Suspicious CSVs while preserving 
       event-level granularity (LSN/USN identifiers)
Input:
- 19 training datasets (LogFile, UsnJrnl, Suspicious CSVs)
- 5 validation datasets (LogFile, UsnJrnl, Suspicious CSVs)

Output:
- data/processed/Phase 1 - Data Cleaning/training_events.csv (1 file)
- data/validation/processed/[dataset]_events.csv (5 files, NO ground truth)
- data/validation/processed/ground_truth/[dataset]_ground_truth.csv (5 files)

Process:
1. Filter LogFile: Keep Time Reversal + Update events
2. Filter UsnJrnl: Keep Basic_Info_Change events
3. Merge with Suspicious CSV by exact LSN/USN match
4. Concatenate LogFile + UsnJrnl events (preserve all)
5. For validation: Drop ground truth labels, save separately


In [39]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np
import os
from pathlib import Path
import gc

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("Phase 1: Event-Level Data Merge")
print("="*80)


Phase 1: Event-Level Data Merge


In [40]:
# Cell 2: Configuration - Dataset Lists

# Training datasets (19 total)
PE_DATASETS = [
    '01-PE', '02-PE', '03-PE', '04-PE', '05-PE', '06-PE',
    '07-PE', '08-PE', '09-PE', '10-PE', '11-PE', '12-PE'
]

APT_TRAINING = [
    '01-APT17', '03-APT21', '04-APT28', '05-APT29', 
    '06-APT30', '07-APT37', '08-APT38', '10-DarkHotel663', 
    '11-DarkHotelbbd', '14-Winnti43b'
]

TRAINING_DATASETS = PE_DATASETS + APT_TRAINING

# Validation datasets (5 total)
VALIDATION_DATASETS = [
    'LoneWolf',     # Lone Wolf (12 files)
    '02-APT19',     # 1 file
    '09-APT40',     # 1 file
    '12-Kimsuky',   # 3 files (FIXED SPELLING)
    '13-Winnti731'  # 1 file
]

print(f"Training datasets: {len(TRAINING_DATASETS)}")
print(f"Validation datasets: {len(VALIDATION_DATASETS)}")

Training datasets: 22
Validation datasets: 5


In [41]:
# Cell 3: Directory Configuration

# Base directories
BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')

# Training data paths
TRAINING_LOGFILE_DIR = BASE_DIR / 'data/training/logfile'
TRAINING_USNJRNL_DIR = BASE_DIR / 'data/training/usnjrnl'
TRAINING_SUSPICIOUS_DIR = BASE_DIR / 'data/training/suspicious'

# Validation data paths
VALIDATION_LOGFILE_DIR = BASE_DIR / 'data/validation/logfile'
VALIDATION_USNJRNL_DIR = BASE_DIR / 'data/validation/usnjrnl'
VALIDATION_SUSPICIOUS_DIR = BASE_DIR / 'data/validation/suspicious'

# Output directories
OUTPUT_DIR = BASE_DIR / 'data/processed/Phase 1 - Data Cleaning'
VALIDATION_OUTPUT_DIR = BASE_DIR / 'data/validation/processed'
VALIDATION_GT_DIR = VALIDATION_OUTPUT_DIR / 'ground_truth'

# Create output directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_GT_DIR.mkdir(parents=True, exist_ok=True)

print("Directories configured")
print(f"Training output: {OUTPUT_DIR}")
print(f"Validation output: {VALIDATION_OUTPUT_DIR}")


Directories configured
Training output: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 1 - Data Cleaning
Validation output: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed


In [42]:
# Cell 4: Helper Functions

def load_dataset_csvs(dataset_name, logfile_dir, usnjrnl_dir, suspicious_dir):
    """
    Load LogFile, UsnJrnl, and Suspicious CSVs for a given dataset.
    
    Parameters:
        dataset_name: Name of dataset (e.g., '01-PE', 'LoneWolf')
        logfile_dir: Directory containing LogFile CSVs
        usnjrnl_dir: Directory containing UsnJrnl CSVs
        suspicious_dir: Directory containing Suspicious CSVs
    
    Returns:
        lf_df, usn_df, sus_df: DataFrames for each CSV type
    """
    # Construct file paths
    if dataset_name == 'LW':
        lf_path = logfile_dir / 'LW-LogFile.csv'
        usn_path = usnjrnl_dir / 'LW-UsnJrnl.csv'
        sus_path = suspicious_dir / 'LW-Suspicious.csv'
    else:
        lf_path = logfile_dir / f'{dataset_name}-LogFile.csv'
        usn_path = usnjrnl_dir / f'{dataset_name}-UsnJrnl.csv'
        sus_path = suspicious_dir / f'{dataset_name}-Suspicious.csv'
    
    # Load CSVs
    lf_df = pd.read_csv(lf_path, low_memory=False)
    usn_df = pd.read_csv(usn_path, low_memory=False)
    sus_df = pd.read_csv(sus_path, low_memory=False)
    
    return lf_df, usn_df, sus_df


def filter_logfile_events(lf_df):
    """
    Filter LogFile to timestamp-relevant events only.
    
    Keep: Time Reversal events + Update events
    
    Parameters:
        lf_df: LogFile DataFrame
    
    Returns:
        Filtered LogFile DataFrame
    """
    # Filter to timestamp-related events
    mask = lf_df['Event'].str.contains(
        'Time Reversal|Update', 
        case=False, 
        na=False
    )
    
    return lf_df[mask].copy()


def filter_usnjrnl_events(usn_df):
    """
    Filter UsnJrnl to timestamp-relevant events only.
    
    Keep: Basic_Info_Change events
    
    Parameters:
        usn_df: UsnJrnl DataFrame
    
    Returns:
        Filtered UsnJrnl DataFrame
    """
    # Filter to Basic_Info_Change events
    mask = usn_df['EventInfo'].str.contains(
        'Basic_Info_Change', 
        case=False, 
        na=False
    )
    
    return usn_df[mask].copy()


def merge_with_suspicious(events_df, suspicious_df, id_column, source_type):
    """
    Merge filtered events with Suspicious CSV by exact LSN/USN match.
    
    Parameters:
        events_df: Filtered LogFile or UsnJrnl DataFrame
        suspicious_df: Suspicious CSV filtered by source
        id_column: Column name for LSN or USN in events_df
        source_type: 'logfile' or 'usnjrnl'
    
    Returns:
        Merged DataFrame with ground truth labels
    """
    # Merge on exact LSN/USN match
    merged = pd.merge(
        events_df,
        suspicious_df[['lsn/usn', 'category', 'detail']],
        left_on=id_column,
        right_on='lsn/usn',
        how='left',
        suffixes=('', '_suspicious')
    )
    
    # Add ground truth label
    merged['is_suspicious'] = merged['category'].notna()
    merged['source_artifact'] = source_type
    
    return merged


def process_single_dataset(dataset_name, logfile_dir, usnjrnl_dir, suspicious_dir, is_validation=False):
    """
    Process a single dataset: Load, filter, merge, concatenate.
    
    Parameters:
        dataset_name: Name of dataset
        logfile_dir, usnjrnl_dir, suspicious_dir: Data directories
        is_validation: If True, drop ground truth labels
    
    Returns:
        DataFrame of merged events, optional ground truth DataFrame
    """
    print(f"\nProcessing: {dataset_name}")
    
    # Load CSVs
    lf_df, usn_df, sus_df = load_dataset_csvs(
        dataset_name, logfile_dir, usnjrnl_dir, suspicious_dir
    )
    
    print(f"  LogFile: {len(lf_df):,} records")
    print(f"  UsnJrnl: {len(usn_df):,} records")
    print(f"  Suspicious: {len(sus_df):,} records")
    
    # Filter events BEFORE merging
    lf_filtered = filter_logfile_events(lf_df)
    usn_filtered = filter_usnjrnl_events(usn_df)
    
    print(f"  LogFile filtered: {len(lf_filtered):,} records ({len(lf_filtered)/len(lf_df)*100:.1f}%)")
    print(f"  UsnJrnl filtered: {len(usn_filtered):,} records ({len(usn_filtered)/len(usn_df)*100:.1f}%)")
    
    # Split Suspicious CSV by source
    sus_lf = sus_df[sus_df['source'] == 'logfile'].copy()
    sus_usn = sus_df[sus_df['source'] == 'usnjrnl'].copy()
    
    # Merge with Suspicious CSV
    lf_merged = merge_with_suspicious(
        lf_filtered, sus_lf, 'LSN', 'logfile'
    )
    
    usn_merged = merge_with_suspicious(
        usn_filtered, sus_usn, 'USN', 'usnjrnl'
    )
    
    # Concatenate LogFile + UsnJrnl events
    events = pd.concat([lf_merged, usn_merged], ignore_index=True)
    
    # Add dataset identifier
    events['dataset'] = dataset_name
    
    print(f"  Total events: {len(events):,}")
    print(f"  Suspicious events: {events['is_suspicious'].sum()}")
    
    # For validation: Extract and drop ground truth
    if is_validation:
        # Extract ground truth columns
        gt_columns = ['LSN', 'USN', 'File/Directory Name', 'FullPath', 
                      'category', 'detail', 'is_suspicious', 'source_artifact']
        
        # Only keep columns that exist
        gt_columns = [col for col in gt_columns if col in events.columns]
        ground_truth = events[gt_columns].copy()
        
        # Drop ground truth from events
        drop_cols = ['category', 'detail', 'is_suspicious', 'lsn/usn']
        drop_cols = [col for col in drop_cols if col in events.columns]
        events = events.drop(columns=drop_cols)
        
        print(f"  Validation mode: Ground truth extracted and removed")
        
        return events, ground_truth
    
    return events, None

print("Helper functions defined")


Helper functions defined


In [43]:
# Cell 5: Process Training Datasets

print("\n" + "="*80)
print("PROCESSING TRAINING DATASETS")
print("="*80)

training_events_list = []

for dataset in TRAINING_DATASETS:
    try:
        events, _ = process_single_dataset(
            dataset,
            TRAINING_LOGFILE_DIR,
            TRAINING_USNJRNL_DIR,
            TRAINING_SUSPICIOUS_DIR,
            is_validation=False
        )
        training_events_list.append(events)
        
        # Clear memory
        gc.collect()
        
    except Exception as e:
        print(f"  ERROR processing {dataset}: {e}")
        continue

# Combine all training datasets
print("\n" + "="*80)
print("COMBINING TRAINING DATASETS")
print("="*80)

training_events = pd.concat(training_events_list, ignore_index=True)

print(f"\nTotal training events: {len(training_events):,}")
print(f"Total suspicious events: {training_events['is_suspicious'].sum():,}")
print(f"Total benign events: {(~training_events['is_suspicious']).sum():,}")

# Save training events
output_path = OUTPUT_DIR / 'training_events.csv'
training_events.to_csv(output_path, index=False)

print(f"\nSaved: {output_path}")
print(f"Size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

# Calculate data reduction by dataset (MB sizes)
print("\n" + "="*80)
print("DATA REDUCTION SUMMARY (FILE SIZES)")
print("="*80)

total_original_mb = 0
total_filtered_mb = 0

# Save individual dataset outputs temporarily to measure size
temp_dir = OUTPUT_DIR / 'temp'
temp_dir.mkdir(exist_ok=True)

for dataset in TRAINING_DATASETS:
    try:
        # Get original file sizes
        if dataset == 'LW':
            lf_path = TRAINING_LOGFILE_DIR / 'LW-LogFile.csv'
            usn_path = TRAINING_USNJRNL_DIR / 'LW-UsnJrnl.csv'
        else:
            lf_path = TRAINING_LOGFILE_DIR / f'{dataset}-LogFile.csv'
            usn_path = TRAINING_USNJRNL_DIR / f'{dataset}-UsnJrnl.csv'
        
        lf_size_mb = lf_path.stat().st_size / 1024 / 1024
        usn_size_mb = usn_path.stat().st_size / 1024 / 1024
        original_mb = lf_size_mb + usn_size_mb
        
        # Save filtered dataset to temp file and measure size
        dataset_events = training_events[training_events['dataset'] == dataset]
        temp_path = temp_dir / f'{dataset}_temp.csv'
        dataset_events.to_csv(temp_path, index=False)
        filtered_mb = temp_path.stat().st_size / 1024 / 1024
        
        reduction = 100 - (filtered_mb / original_mb * 100)
        
        total_original_mb += original_mb
        total_filtered_mb += filtered_mb
        
        print(f"{dataset:20s} LogFile: {lf_size_mb:6.2f} MB | UsnJrnl: {usn_size_mb:6.2f} MB | After: {filtered_mb:6.2f} MB | Reduction: {reduction:5.1f}%")
        
        # Clean up temp file
        temp_path.unlink()
        
    except Exception as e:
        print(f"  ERROR calculating reduction for {dataset}: {e}")
        continue

# Clean up temp directory
temp_dir.rmdir()

overall_reduction = 100 - (total_filtered_mb / total_original_mb * 100)
print("="*80)
print(f"{'TOTAL':20s} Original: {total_original_mb:6.2f} MB | After: {total_filtered_mb:6.2f} MB | Reduction: {overall_reduction:5.1f}%")
print("="*80)



PROCESSING TRAINING DATASETS

Processing: 01-PE
  LogFile: 39,077 records
  UsnJrnl: 316,817 records
  Suspicious: 4 records
  LogFile filtered: 1,235 records (3.2%)
  UsnJrnl filtered: 24,002 records (7.6%)
  Total events: 25,237
  Suspicious events: 2

Processing: 02-PE
  LogFile: 14,783 records
  UsnJrnl: 247,386 records
  Suspicious: 3 records
  LogFile filtered: 97 records (0.7%)
  UsnJrnl filtered: 16,959 records (6.9%)
  Total events: 17,056
  Suspicious events: 1

Processing: 03-PE
  LogFile: 24,063 records
  UsnJrnl: 245,425 records
  Suspicious: 4 records
  LogFile filtered: 97 records (0.4%)
  UsnJrnl filtered: 16,880 records (6.9%)
  Total events: 16,977
  Suspicious events: 2

Processing: 04-PE
  LogFile: 12,731 records
  UsnJrnl: 263,451 records
  Suspicious: 58 records
  LogFile filtered: 79 records (0.6%)
  UsnJrnl filtered: 4,949 records (1.9%)
  Total events: 5,028
  Suspicious events: 2

Processing: 05-PE
  LogFile: 14,242 records
  UsnJrnl: 265,287 records
  Suspic

In [44]:
# Cell 6: Process Validation Datasets

print("\n" + "="*80)
print("PROCESSING VALIDATION DATASETS")
print("="*80)

for dataset in VALIDATION_DATASETS:
    try:
        events, ground_truth = process_single_dataset(
            dataset,
            VALIDATION_LOGFILE_DIR,
            VALIDATION_USNJRNL_DIR,
            VALIDATION_SUSPICIOUS_DIR,
            is_validation=True
        )
        
        # Save validation events (NO ground truth)
        events_path = VALIDATION_OUTPUT_DIR / f'{dataset}_events.csv'
        events.to_csv(events_path, index=False)
        print(f"  Saved events: {events_path}")
        
        # Save ground truth separately
        gt_path = VALIDATION_GT_DIR / f'{dataset}_ground_truth.csv'
        ground_truth.to_csv(gt_path, index=False)
        print(f"  Saved ground truth: {gt_path}")
        
        # Clear memory
        gc.collect()
        
    except Exception as e:
        print(f"  ERROR processing {dataset}: {e}")
        continue

# Validation data reduction summary (MB sizes)
print("\n" + "="*80)
print("VALIDATION DATA REDUCTION SUMMARY (FILE SIZES)")
print("="*80)

total_original_val_mb = 0
total_filtered_val_mb = 0

for dataset in VALIDATION_DATASETS:
    try:
        # Get original file sizes
        if dataset == 'LoneWolf':
            lf_path = VALIDATION_LOGFILE_DIR / 'LoneWolf-LogFile.csv'
            usn_path = VALIDATION_USNJRNL_DIR / 'LoneWolf-UsnJrnl.csv'
        else:
            lf_path = VALIDATION_LOGFILE_DIR / f'{dataset}-LogFile.csv'
            usn_path = VALIDATION_USNJRNL_DIR / f'{dataset}-UsnJrnl.csv'
        
        lf_size_mb = lf_path.stat().st_size / 1024 / 1024
        usn_size_mb = usn_path.stat().st_size / 1024 / 1024
        original_mb = lf_size_mb + usn_size_mb
        
        # Get filtered file size
        events_path = VALIDATION_OUTPUT_DIR / f'{dataset}_events.csv'
        filtered_mb = events_path.stat().st_size / 1024 / 1024
        
        reduction = 100 - (filtered_mb / original_mb * 100)
        
        total_original_val_mb += original_mb
        total_filtered_val_mb += filtered_mb
        
        print(f"{dataset:20s} LogFile: {lf_size_mb:6.2f} MB | UsnJrnl: {usn_size_mb:6.2f} MB | After: {filtered_mb:6.2f} MB | Reduction: {reduction:5.1f}%")
        
    except Exception as e:
        print(f"  ERROR calculating reduction for {dataset}: {e}")
        continue

overall_reduction_val = 100 - (total_filtered_val_mb / total_original_val_mb * 100)
print("="*80)
print(f"{'TOTAL':20s} Original: {total_original_val_mb:6.2f} MB | After: {total_filtered_val_mb:6.2f} MB | Reduction: {overall_reduction_val:5.1f}%")
print("="*80)



PROCESSING VALIDATION DATASETS

Processing: LoneWolf
  LogFile: 16,882 records
  UsnJrnl: 352,849 records
  Suspicious: 15 records
  LogFile filtered: 587 records (3.5%)
  UsnJrnl filtered: 33,713 records (9.6%)
  Total events: 34,300
  Suspicious events: 12
  Validation mode: Ground truth extracted and removed
  Saved events: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed/LoneWolf_events.csv
  Saved ground truth: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed/ground_truth/LoneWolf_ground_truth.csv

Processing: 02-APT19
  LogFile: 27,718 records
  UsnJrnl: 325,721 records
  Suspicious: 3 records
  LogFile filtered: 338 records (1.2%)
  UsnJrnl filtered: 23,465 records (7.2%)
  Total events: 23,803
  Suspicious events: 2
  Validation mode: Ground truth extracted and removed
  Saved events: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed/02-APT19_events.csv
  Saved ground truth: /Users/soni/Github/Digital-Detectives_

In [45]:
# Cell 7: Validation Summary

print("\n" + "="*80)
print("PHASE 1 COMPLETE")
print("="*80)

print("\nTraining Data:")
print(f"  Datasets processed: {len(TRAINING_DATASETS)}")
print(f"  Total events: {len(training_events):,}")
print(f"  Suspicious events: {training_events['is_suspicious'].sum():,}")
print(f"  Data reduction: {100 - (len(training_events) / training_events.groupby('dataset').size().sum() * 100):.1f}%")

print("\nValidation Data:")
print(f"  Datasets processed: {len(VALIDATION_DATASETS)}")
for dataset in VALIDATION_DATASETS:
    events_path = VALIDATION_OUTPUT_DIR / f'{dataset}_events.csv'
    if events_path.exists():
        df = pd.read_csv(events_path)
        print(f"  {dataset}: {len(df):,} events (no ground truth labels)")

print("\nOutput Files:")
print(f"  Training: {OUTPUT_DIR / 'training_events.csv'}")
print(f"  Validation events: {VALIDATION_OUTPUT_DIR}")
print(f"  Validation ground truth: {VALIDATION_GT_DIR}")

print("\nNext: Phase 2 - Feature Engineering")



PHASE 1 COMPLETE

Training Data:
  Datasets processed: 22
  Total events: 373,114
  Suspicious events: 288
  Data reduction: 0.0%

Validation Data:
  Datasets processed: 5
  LoneWolf: 34,300 events (no ground truth labels)
  02-APT19: 23,803 events (no ground truth labels)
  09-APT40: 23,519 events (no ground truth labels)
  12-Kimsuky: 17,521 events (no ground truth labels)
  13-Winnti731: 14,182 events (no ground truth labels)

Output Files:
  Training: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 1 - Data Cleaning/training_events.csv
  Validation events: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed
  Validation ground truth: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed/ground_truth

Next: Phase 2 - Feature Engineering


In [46]:
# Cell 8: Quick Data Inspection

print("\n" + "="*80)
print("TRAINING DATA SAMPLE")
print("="*80)

print("\nColumns:", training_events.columns.tolist())
print(f"\nShape: {training_events.shape}")

print("\nSuspicious Events Sample:")
print(training_events[training_events['is_suspicious'] == True].head(3))

print("\nBenign Events Sample:")
print(training_events[training_events['is_suspicious'] == False].head(3))

print("\nSource Artifact Distribution:")
print(training_events['source_artifact'].value_counts())

print("\nDataset Distribution:")
print(training_events['dataset'].value_counts().sort_index())



TRAINING DATA SAMPLE

Columns: ['LSN', 'EventTime(UTC+8)', 'Event', 'Detail', 'File/Directory Name', 'Full Path', 'CreationTime', 'ModifiedTime', 'MFTModifiedTime', 'AccessedTime', 'Redo', 'Target VCN', 'Cluster Index', 'lsn/usn', 'category', 'detail', 'is_suspicious', 'source_artifact', 'TimeStamp(UTC+8)', 'USN', 'FullPath', 'EventInfo', 'SourceInfo', 'FileAttribute', 'Carving Flag', 'FileReferenceNumber', 'ParentFileReferenceNumber', 'dataset']

Shape: (373114, 28)

Suspicious Events Sample:
                LSN EventTime(UTC+8)                Event  \
1230   8.730038e+09              NaN  Time Reversal Event   
25191           NaN              NaN                  NaN   
25333  1.005554e+10              NaN  Time Reversal Event   

                                                  Detail  \
1230   CreationTime : 2023-12-23 00:21:36 -> 2022-12-...   
25191                                                NaN   
25333  ModifiedTime : 2023-12-26 15:16:49 -> 2022-12-...   

              